# Texture Mixer coupling at n=1,000 with trace-corrected KID (Table 1 TM block)

Renders `tm_scale1000/tmc1000/{lerp,mean,ot_r0.03}` and scores Table 1's Texture Mixer block (`TMk*` macros) with the corrected KID reference term. Cell 7 (pair retrieval) crashed and its human-study/prompt-recovery cells are absent from this saved copy.

**Needs:** DTD, the Texture Mixer checkpoint.

**Maintained runnable path:** `experiments/tm_1000.py` reproduces this render; `score_only.py` rescores it (`Table1-TM-1000`). Kept for provenance.

# Pre-submission additions from the simulated review

| # | what | phase | output |
|---|---|---|---|
| 1 | **Texture Mixer coupling at n = 1,000**: native lerp / target-mean / entropic (ε = 0.03×median, Sinkhorn to marginal error < 1e-3), same encodings and generator | TF1 (cells 1–3) | `tm_scale1000/tmc1000/<rule>/`, `tm_coupling1000_stats.csv` |
| 2 | **Canonical scoring + paired jackknife ΔKID** for those 1,000 pairs (trace-corrected KID) | torch (cell 5) | `jackknife/tm1000_*.csv`, macro lines |
| 3 | **Pair-conditioned diagnostic**: pair retrieval — does a midpoint retrieve its *own* endpoint pair among all pairs? MRR / top-1 for every method on the 120-pair table and the matched-rendering rows, with source-copy / target-copy / target-mean controls | torch (cell 6) | `jackknife/pair_retrieval_*.csv`, macro lines |
| 4 | **GPT prompt recovery**: searches every notebook and JSON on the Drive for the prompt strings and call ledger | torch (cell 7) | printed candidates |
| 5 | **Human-study stimuli**: 2AFC panels (A · candidate · candidate · B, sides randomised) for AAT vs field-linear / hard / Texture Mixer on 100 pairs, plus the answer key — ready to upload the day the discussion period opens | torch (cell 8) | `human_study/` |

Run cells 1–3, **Runtime → Restart session**, then cells 4–8. Every loop is resumable.


In [ ]:
# ============================================================
# PHASE 1 (TensorFlow), CELL 1 — Texture Mixer setup, verbatim. No torch in this phase.
# ============================================================
import os, sys, types, importlib, importlib.util, importlib.machinery, glob
import numpy as np
from PIL import Image, ImageFile
import matplotlib.pyplot as plt
ImageFile.LOAD_TRUNCATED_IMAGES = True

from google.colab import drive
drive.mount('/content/drive')
BASE = os.environ.get('DATA_ROOT', '/content/data')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
TM_STORE = f'{BASE}/texturemixer'
TM_OUT = f'{TM_STORE}/midpoints'
os.makedirs(TM_OUT, exist_ok=True)

# --- repo ---
os.chdir('/content')
if not os.path.isdir('/content/TextureMixer'):
    !git clone -q https://github.com/ningyu1991/TextureMixer.git
if not os.path.isdir('/content/TextureMixer/tensorflow_vgg'):
    !git clone -q https://github.com/machrisaa/tensorflow-vgg.git /content/TextureMixer/tensorflow_vgg
os.chdir('/content/TextureMixer'); sys.path.insert(0, '/content/TextureMixer')

# --- Python 3.12 removed `imp` ---
if 'imp' not in sys.modules:
    _imp = types.ModuleType('imp')
    def _load_source(name, pathname, file=None):
        spec = importlib.util.spec_from_file_location(name, pathname)
        mod = importlib.util.module_from_spec(spec); sys.modules[name] = mod
        spec.loader.exec_module(mod); return mod
    def _find_module(name, path=None):
        spec = importlib.machinery.PathFinder().find_spec(name, path)
        if spec is None: raise ImportError(name)
        return (None, spec.origin, ('.py','r',1))
    _imp.load_source=_load_source; _imp.find_module=_find_module
    _imp.new_module=lambda n: types.ModuleType(n); _imp.reload=importlib.reload
    _imp.acquire_lock=lambda: None; _imp.release_lock=lambda: None; _imp.PY_SOURCE=1
    sys.modules['imp'] = _imp
    print('imp shim installed')

# --- TF1 graph mode on TF2 ---
import tensorflow as _tf2
print('TF', _tf2.__version__, '| GPU:', _tf2.config.list_physical_devices('GPU'))
tf1 = _tf2.compat.v1
tf1.disable_eager_execution(); tf1.disable_v2_behavior()
tf1.logging.set_verbosity(tf1.logging.ERROR)
if not hasattr(tf1, 'Dimension'): tf1.Dimension = getattr(_tf2, 'Dimension', int)

!pip install -q tf_slim
import tf_slim as _slim
_contrib = types.ModuleType('tensorflow.contrib')
_contrib.slim=_slim; _contrib.framework=_slim; _contrib.layers=_slim
sys.modules['tensorflow.contrib']=_contrib
sys.modules['tensorflow.contrib.slim']=_slim
sys.modules['tensorflow.contrib.framework']=_slim
sys.modules['tensorflow.contrib.layers']=_slim
tf1.contrib=_contrib
sys.modules['tensorflow'] = tf1                 # repo does `import tensorflow as tf`
print('tf_slim wired into tf.contrib')

# --- Python-3 syntax fixes in the repo ---
_p='/content/TextureMixer/tfutil.py'; _s=open(_p).read()
_s=_s.replace('v.shape.ndims is 0','v.shape.ndims == 0').replace('v.shape.ndims is 1','v.shape.ndims == 1')
open(_p,'w').write(_s)

# --- checkpoint (cached on Drive) ---
CKPT = f'{TM_STORE}/earth_texture/network-final.pkl'
if not os.path.exists(CKPT):
    !pip install -q gdown
    !gdown -q 1ObAFBPGaRJFo11LUa0qNhRX14nTEWKC1 -O /content/earth.zip
    !cd /content && unzip -oq earth.zip -d "{TM_STORE}"
print('checkpoint present:', os.path.exists(CKPT))

# --- load networks (9-tuple, bind by name) ---
import tfutil, misc
sess = tfutil.create_session(force_as_default=True)
obj = misc.load_pkl(CKPT)
nets = {getattr(o,'name',f'_{i}'): o for i,o in enumerate(obj)}
print('components:', list(nets))
Es_zg, Es_zl, Gs = nets['Es_zg'], nets['Es_zl'], nets['Gs']
print('LOADED OK — Gs', Gs.input_shapes, '->', Gs.output_shapes)

In [ ]:
# ============================================================
# PHASE 1, CELL 2 — Texture Mixer encode/generate, verbatim
# ============================================================
import random
TM_RES = 128
LATENT_IDX = 0          # 0 = mean (verified); 1 = second moment

def tm_prep(pil):
    a = np.asarray(pil.convert('RGB').resize((TM_RES,)*2), dtype=np.float32)
    return (a/127.5 - 1.0).transpose(2,0,1)[None]

def tm_to_pil(arr):
    return Image.fromarray(np.clip((arr[0].transpose(1,2,0)+1.0)*127.5,0,255).astype(np.uint8))

def tm_encode(pil):
    x = tm_prep(pil)
    return (Es_zg.run(x, return_as_list=True)[LATENT_IDX],
            Es_zl.run(x, return_as_list=True)[LATENT_IDX])

def tm_generate(zg, zl):
    zg_t = np.tile(zg, (1,1,zl.shape[2],zl.shape[3]))
    return tm_to_pil(Gs.run(zg_t, zl, return_as_list=True)[0])

def texturemixer_interp(A, B, weight=0.5):
    zgA, zlA = tm_encode(A); zgB, zlB = tm_encode(B)
    return tm_generate((1-weight)*zgA + weight*zgB, (1-weight)*zlA + weight*zlB)

# --- DTD (a few reads, with retry for Drive EIO) ---
DTD_ROOT = f'{BASE}/dtd/images'
cats = sorted([c for c in os.listdir(DTD_ROOT) if os.path.isdir(f'{DTD_ROOT}/{c}')])
def dtd_path(c,s): return random.Random(s).choice(sorted(glob.glob(f'{DTD_ROOT}/{c}/*.jpg')))
def safe_open(p, tries=4):
    import time as _t
    for i in range(tries):
        try:
            im = Image.open(p); im.load(); return im.convert('RGB')
        except OSError:
            if i == tries-1: raise
            _t.sleep(0.5*(i+1))
print(f'{len(cats)} categories')

# --- reconstruction sanity check ---
probe = safe_open(dtd_path(cats[20], 7))
zg, zl = tm_encode(probe)
fig, ax = plt.subplots(1,2, figsize=(6,3.2))
ax[0].imshow(probe.resize((TM_RES,)*2)); ax[0].set_title('input'); ax[0].axis('off')
ax[1].imshow(tm_generate(zg,zl)); ax[1].set_title('Gs(E(input))'); ax[1].axis('off')
plt.tight_layout(); plt.show()
print('>>> if the right panel resembles the left, continue to cell 3')

In [ ]:
# ============================================================
# PHASE 1, CELL 3 — Texture Mixer coupling at n = 1,000 (lerp / mean / entropic eps_rel = 0.03). Same code as the 120-pair ablation.
# ============================================================
import numpy as np, pandas as pd, time
OUT_ROOT=f'{BASE}/tm_scale1000'; TMC=f'{OUT_ROOT}/tmc1000'; EPS_REL=0.03; RULES=['lerp','mean',f'ot_r{EPS_REL}']
for r_ in RULES: os.makedirs(f'{TMC}/{r_}',exist_ok=True)
KS1000=sorted(int(os.path.basename(f)[:4]) for f in glob.glob(f'{OUT_ROOT}/A/*.png') if os.path.exists(f'{OUT_ROOT}/B/{os.path.basename(f)}')); print(len(KS1000),'pairs')
def lse(M,ax): mx=M.max(ax,keepdims=True); return (mx+np.log(np.exp(M-mx).sum(ax,keepdims=True))).squeeze(ax)
def sink_log(C,eps,tol=1e-3,max_it=5000):
    n,m=C.shape; la,lb=-np.log(n),-np.log(m); f=np.zeros(n); g=np.zeros(m); it=0
    while it<max_it:
        for _ in range(50): f=eps*(la-lse((g[None,:]-C)/eps,1)); g=eps*(lb-lse((f[:,None]-C)/eps,0)); it+=1
        P=np.exp((f[:,None]+g[None,:]-C)/eps); err=max(np.abs(P.sum(1)*n-1).max(),np.abs(P.sum(0)*m-1).max())
        if err<tol: break
    return P,float(err),it
def enc_np(pil): x=tm_prep(pil); return Es_zg.run(x,return_as_list=True)[LATENT_IDX], Es_zl.run(x,return_as_list=True)[LATENT_IDX]
def gen_np(zg,zl): zg_t=np.tile(zg,(1,1,zl.shape[2],zl.shape[3])); return tm_to_pil(Gs.run(zg_t,zl,return_as_list=True)[0])
if 'tm_to_pil' not in globals():
    def tm_to_pil(arr): a=np.asarray(arr); a=a[0] if a.ndim==4 else a; return Image.fromarray(np.clip((a.transpose(1,2,0)+1)*127.5,0,255).astype(np.uint8))
CSV=f'{BASE}/tm_coupling1000_stats.csv'; rows=pd.read_csv(CSV).to_dict('records') if os.path.exists(CSV) else []; have={(r['k'],r['rule']) for r in rows}; t0=time.time()
for n_,k in enumerate(KS1000):
    if all((k,r_) in have and os.path.exists(f'{TMC}/{r_}/{k:04d}.png') for r_ in RULES): continue
    A=Image.open(f'{OUT_ROOT}/A/{k:04d}.png').convert('RGB'); B=Image.open(f'{OUT_ROOT}/B/{k:04d}.png').convert('RGB')
    (gA,lA),(gB,lB)=enc_np(A),enc_np(B); C_,H,W=lA.shape[1:]; X=lA[0].reshape(C_,-1).T.astype(np.float64); Y=lB[0].reshape(C_,-1).T.astype(np.float64)
    Cm=((X[:,None,:]-Y[None,:,:])**2).sum(-1); medC=float(np.median(Cm)); zg_mid=0.5*(gA+gB); varY=Y.var(0).sum()
    P,err,it=sink_log(Cm,EPS_REL*medC); U=(P/P.sum(1,keepdims=True))@Y
    T={'lerp':(Y,dict()),'mean':(np.tile(Y.mean(0),(len(X),1)),dict()),f'ot_r{EPS_REL}':(U,dict(marg_err=err,iters=it))}
    for name,(Ut,info) in T.items():
        if (k,name) not in have: rows.append(dict(k=k,rule=name,med_cost=medC,var_ret=float(Ut.var(0).sum()/varY),**info))
        fp=f'{TMC}/{name}/{k:04d}.png'
        if not os.path.exists(fp): gen_np(zg_mid,(0.5*X+0.5*Ut).T.reshape(1,C_,H,W).astype(np.float32)).save(fp)
    if n_%50==0: pd.DataFrame(rows).to_csv(CSV,index=False); print(f'{n_+1}/{len(KS1000)} {time.time()-t0:.0f}s')
pd.DataFrame(rows).to_csv(CSV,index=False); S=pd.DataFrame(rows); print(S.groupby('rule')[['var_ret']].median().round(3)); print('max marginal error',S.marg_err.max() if 'marg_err' in S else None)
print('>>> PHASE 1 done. Runtime -> Restart session, then run cells 4-8.')


## Restart the runtime here (Runtime → Restart session), then run cells 4–8.


In [ ]:
# ============================================================
# PHASE 2 (torch), CELL 4 — fast canonical SETUP — canonical protocol (finish_day / mechanism cell 0, verbatim) + pinned CNT commit
# ============================================================
import os,sys,glob,random,time,gc,math,json,zipfile,hashlib,itertools
import numpy as np, pandas as pd, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageFile
from scipy.stats import wilcoxon, spearmanr
from scipy.optimize import linear_sum_assignment
from scipy import linalg
from google.colab import drive
drive.mount('/content/drive')
ImageFile.LOAD_TRUNCATED_IMAGES=True; device='cuda'
assert torch.cuda.is_available(),'switch runtime to GPU'
BASE=os.environ.get('DATA_ROOT', '/content/data')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
SCALE=f'{BASE}/tm_scale1000'; OUT=f'{BASE}/matching_ablation'
MECH=f'{BASE}/mechanism'; os.makedirs(MECH,exist_ok=True); FIGS=f'{MECH}/figs'; os.makedirs(FIGS,exist_ok=True)
TMP='/content/tmp_imgs'; os.makedirs(TMP,exist_ok=True)
t0=time.time()

# --- DTD ---
LOCAL='/content/dtd_images'; ZIP=f'{BASE}/dtd_images.zip'
if len(glob.glob(f'{LOCAL}/*/*.jpg'))<5000:
    os.system('rm -rf /content/_dtd'); os.system(f'unzip -qo "{ZIP}" -d /content/_dtd')
    inner=os.listdir('/content/_dtd'); src='/content/_dtd/images' if 'images' in inner else f'/content/_dtd/{inner[0]}'
    os.system(f'rm -rf {LOCAL} && mv "{src}" {LOCAL}')
cats=sorted(os.listdir(LOCAL)); print(f'DTD ready: {len(cats)} cats, {time.time()-t0:.0f}s')
os.system('pip install -q timm lpips einops 2>/dev/null')

# --- CNT ---
os.chdir('/content'); os.system('rm -rf cnt-siga24; git clone -q https://github.com/phtu-cs/compositional-neural-textures.git cnt-siga24 && cd cnt-siga24 && git checkout -q 07269302610c92ce38e2dbbe31c492fd09e1c73e')
os.system(f'rm -rf /content/cnt-siga24/ckpts; ln -sfn "{BASE}/cnt_ckpts" /content/cnt-siga24/ckpts')
CNT='/content/cnt-siga24'; os.chdir(CNT); sys.path.insert(0,CNT)
try:
    from src.inference.core import TexAutoEncoderInference, load_image
except Exception as e:
    print('installing missing deps:',type(e).__name__)
    os.system('pip install -q -r requirements.in; pip install -q "git+https://github.com/facebookresearch/detectron2.git"')
    from src.inference.core import TexAutoEncoderInference, load_image
model=TexAutoEncoderInference(ckpt_path=Path(f'{CNT}/ckpts/finetuned-model256.ckpt'),device=device)
IMG_SIZE=256; FMS=[round(IMG_SIZE*r) for r in model.generator.feature_map_size_ratios]
EPS,ITERS,NSUB=0.05,100,4096
CNT_COMMIT=os.popen(f'cd {CNT} && git rev-parse HEAD').read().strip()
print(f'model ready {time.time()-t0:.0f}s | FMS={FMS} | CNT commit {CNT_COMMIT[:10]}')

_tc=[0]
def cnt_load(x,size=IMG_SIZE):
    if isinstance(x,(str,Path)): return load_image(Path(x),size).to(device)
    _tc[0]+=1; fp=f'{TMP}/t{_tc[0]%64}.png'; x.convert('RGB').save(fp); return load_image(Path(fp),size).to(device)
def to_pil(dec):
    a=((dec.squeeze(0).detach().cpu().clamp(-1,1)+1)/2).permute(1,2,0).numpy(); return Image.fromarray((a*255).astype(np.uint8))
def get_gen_inputs(img):
    with torch.no_grad():
        blob=model.encode(img); L=model.splat(blob)
        return blob,L["layer_level_layouts"][-1][0]["blob_features"].clone(),L["layer_level_layouts"][-1][0]["score_img"].clone()
def _pyr(si):
    sp={}; s=si; sp[s.shape[-1]]=s
    while s.shape[-1]>min(FMS): s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
    return sp
def build_style_maps(bf,si,fms):
    sp=_pyr(si); return {size: torch.einsum("bnc,bnhw->bchw",bf,sp[size]) for size in set(fms)}
def decode_with_style_maps(blob_ref,sm):
    with torch.no_grad():
        L=model.splat(blob_ref); si=L["layer_level_layouts"][-1][0]["score_img"]
        fms=[round(si.shape[-1]*r) for r in model.generator.feature_map_size_ratios]; sp=_pyr(si)
        fi=sp[fms[0]][:,1:].amax(dim=1)[:,None].repeat(1,model.generator.n_embedding,1,1); x=fi.clone()
        for i in range(len(fms)):
            x=model.generator.spade_blocks[i](x,sm[fms[i]])
            if i==len(fms)-1: break
            elif fms[i]!=fms[i+1]: x=F.interpolate(x,size=x.shape[-1]*2,mode="bilinear",align_corners=False)
        return torch.tanh(x.unsqueeze(1)).flatten(1,2)
def decode_stagewise(blob_ref,sm_by_stage):
    """same decoder, but stage i receives sm_by_stage[i] (dict size->map). For E5 per-stage swaps."""
    with torch.no_grad():
        L=model.splat(blob_ref); si=L["layer_level_layouts"][-1][0]["score_img"]
        fms=[round(si.shape[-1]*r) for r in model.generator.feature_map_size_ratios]; sp=_pyr(si)
        fi=sp[fms[0]][:,1:].amax(dim=1)[:,None].repeat(1,model.generator.n_embedding,1,1); x=fi.clone()
        for i in range(len(fms)):
            x=model.generator.spade_blocks[i](x,sm_by_stage[i][fms[i]])
            if i==len(fms)-1: break
            elif fms[i]!=fms[i+1]: x=F.interpolate(x,size=x.shape[-1]*2,mode="bilinear",align_corners=False)
        return torch.tanh(x.unsqueeze(1)).flatten(1,2)
def build_d(ia,ib):
    ba,bfa,sia=get_gen_inputs(ia); bb,bfb,sib=get_gen_inputs(ib)
    return {"blob_a":ba,"bf_a":bfa,"si_a":sia,"blob_b":bb,"bf_b":bfb,"si_b":sib}
def _grids(d):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS); sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    L=max(FMS); A,B=sa[L][0],sb[L][0]; Cc,H,W=A.shape
    return L,Cc,H,W,A.permute(1,2,0).reshape(-1,Cc),B.permute(1,2,0).reshape(-1,Cc)
_TAG={'s':1,'t':2}
def _sub(N,seed,tag):
    g=torch.Generator(device=device).manual_seed(int(seed)*7919+_TAG[tag]); return torch.randperm(N,generator=g,device=device)[:NSUB]
def sm_from_field(Z,L,H,W,Cc):
    it=Z.reshape(H,W,Cc).permute(2,0,1).unsqueeze(0)
    return {s:(it if s==L else F.interpolate(it,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)}
def _dec(d,Z,L,H,W,Cc,layout='a'): return decode_with_style_maps(d["blob_"+layout],sm_from_field(Z.float(),L,H,W,Cc))
def interp_lin_pixel(d,eta):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS); sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    return decode_with_style_maps(d["blob_a"],{s:(1-eta)*sa[s]+eta*sb[s] for s in set(FMS)})
def make_recon(pil):
    e=cnt_load(pil); return interp_lin_pixel(build_d(e,e),0.0)
def pair_imgs(k): return Image.open(f'{SCALE}/A/{k:04d}.png').convert('RGB'), Image.open(f'{SCALE}/B/{k:04d}.png').convert('RGB')

# --- canonical Sinkhorn (as used in the paper) + converged log-domain Sinkhorn with duals ---
def sinkhorn_plan(x,y,eps,iters=ITERS):
    Cm=torch.cdist(x,y).pow(2); K=torch.exp(-Cm/eps)
    u=torch.ones(len(x),device=x.device)/len(x); v=torch.ones(len(y),device=y.device)/len(y)
    a=torch.ones_like(u); b=torch.ones_like(v)
    for _ in range(iters): a=u/(K@b+1e-30); b=v/(K.T@a+1e-30)
    return a[:,None]*K*b[None,:], Cm
def sinkhorn_log(x,y,eps,iters=3000,tol=1e-6):
    """balanced, uniform marginals, float64 log-domain; returns P, duals f,g, cost, marginal residual, iterations."""
    X=x.double(); Y=y.double(); Cm=torch.cdist(X,Y).pow(2); n,m=Cm.shape
    la=-math.log(n); lb=-math.log(m); f=torch.zeros(n,device=X.device,dtype=torch.float64); g=torch.zeros(m,device=X.device,dtype=torch.float64)
    for it in range(1,iters+1):
        f=eps*(la-torch.logsumexp((g[None,:]-Cm)/eps,1)); g=eps*(lb-torch.logsumexp((f[:,None]-Cm)/eps,0))
        if it%10==0:
            logP=(f[:,None]+g[None,:]-Cm)/eps; res=float((torch.exp(logP).sum(1)-1/n).abs().max()*n)
            if res<tol: break
    logP=(f[:,None]+g[None,:]-Cm)/eps; P=torch.exp(logP)
    res=float(max((P.sum(1)-1/n).abs().max()*n,(P.sum(0)-1/m).abs().max()*m))
    return P,f,g,Cm,res,it
def plan_stats(P):
    P=P.double(); n=P.shape[0]; q=P/(P.sum(1,keepdim=True)+1e-300)
    Hrow=float(-(q*torch.log(q+1e-300)).sum(1).mean()); Htot=float(-(P*torch.log(P+1e-300)).sum())
    return dict(H_row=Hrow,H_total=Htot,MI_nats=math.log(n)-Hrow)
def natural_extension(x_all,y,g,eps,chunk=1024):
    """T(x)=sum_j q_j(x) y_j, q_j ∝ exp((g_j-||x-y_j||²)/eps) — dual-potential extension of a converged plan."""
    out=[]
    for i in range(0,len(x_all),chunk):
        Cm=torch.cdist(x_all[i:i+chunk].double(),y.double()).pow(2); q=torch.softmax((g[None,:]-Cm)/eps,1); out.append(q@y.double())
    return torch.cat(out)

# --- matching rules -> sampled targets (M,d); full field via canonical nn extension ---
def _targets(sf,tf,rule,seed,eps=EPS,M=NSUB,n_exact=1024,plan_kind='canonical'):
    seed=int(seed)   # numpy int64 from pandas breaks torch manual_seed
    N=sf.shape[0]; i_s=_sub(N,seed,'s')[:M]; i_t=_sub(N,seed,'t')[:M]; src,tgt=sf[i_s],tf[i_t]; info={}
    if rule in('ot','ot_sharp','ot_soft'):
        e={'ot':eps,'ot_sharp':0.005,'ot_soft':0.5}[rule] if rule!='ot' else eps
        if plan_kind=='canonical': P,_=sinkhorn_plan(src,tgt,e); info=dict(eps=e,**plan_stats(P))
        else: P,f,g,_,res,it=sinkhorn_log(src,tgt,e); info=dict(eps=e,res=res,iters=it,**plan_stats(P)); info['g']=g
        tt=((P/(P.sum(1,keepdim=True)+1e-30))@tgt.double()).float() if P.dtype==torch.float64 else (P/(P.sum(1,keepdim=True)+1e-30))@tgt
    elif rule=='exact':
        g_=torch.Generator(device=device).manual_seed(seed+11)
        js=torch.randperm(M,generator=g_,device=device)[:n_exact]; is_=torch.randperm(M,generator=g_,device=device)[:n_exact]
        r,c=linear_sum_assignment(torch.cdist(src[is_],tgt[js]).pow(2).cpu().numpy())
        tt=torch.full_like(src,float('nan')); tt[is_[torch.as_tensor(r,device=device)]]=tgt[js[torch.as_tensor(c,device=device)]]
        ok=~torch.isnan(tt[:,0]); tt[~ok]=tt[ok][torch.cdist(src[~ok],src[ok]).argmin(1)]; info=dict(n_exact=n_exact)
    elif rule=='nn': tt=tgt[torch.cdist(src,tgt).argmin(1)]
    elif rule=='gauss':
        X,Y=src.double(),tgt.double(); mx,my=X.mean(0),Y.mean(0); Xc=X-mx; Yc=Y-my; Cc=X.shape[1]; I=torch.eye(Cc,device=device,dtype=torch.float64); ridge=1e-4
        Sa=(Xc.T@Xc)/(len(X)-1)+ridge*I; Sb=(Yc.T@Yc)/(len(Y)-1)+ridge*I
        def ps(M_,fl):
            e_,V=torch.linalg.eigh(.5*(M_+M_.T)); e_=e_.clamp_min(fl); return (V*e_.sqrt())@V.T
        def pis(M_,fl):
            e_,V=torch.linalg.eigh(.5*(M_+M_.T)); e_=e_.clamp_min(fl); return (V*e_.rsqrt())@V.T
        Sh=ps(Sa,ridge); Sih=pis(Sa,ridge); Amap=Sih@ps(Sh@Sb@Sh,0.0)@Sih; Amap=.5*(Amap+Amap.T)
        tt=(my+Xc@Amap.T).float(); info=dict(A=Amap,mu=(mx,my))
    elif rule=='uniform': tt=tgt.mean(0,keepdim=True).expand_as(src).clone()
    elif rule=='random':
        g_=torch.Generator(device=device).manual_seed(seed+7); tt=tgt[torch.randperm(M,generator=g_,device=device)]
    elif rule=='meanshift': tt=src+(tgt.mean(0)-src.mean(0))      # CNT-style native mean shift, adapted to the interpolation contract
    else: raise ValueError(rule)
    return i_s,tt,info
def field(d,rule,seed,**kw):
    seed=int(seed)
    """full-grid target field U (N,d) with the canonical nn extension; also returns X,Y grids and plan info."""
    L,Cc,H,W,sf,tf=_grids(d); i_s,tt,info=_targets(sf,tf,rule,seed,**kw)
    nn=torch.cdist(sf,sf[i_s]).argmin(1); return dict(U=tt[nn],X=sf,Y=tf,L=L,Cc=Cc,H=H,W=W,i_s=i_s,tt=tt,nn=nn,info=info)
def zt(fd,t): return (1-t)*fd['X']+t*fd['U']
def decode_field(d,fd,Z): return _dec(d,Z,fd['L'],fd['H'],fd['W'],fd['Cc'])

# --- fixed-layout subspace of A: X = W_A F_A ---
def splat_W(d,side='a'):
    sp=_pyr(d['si_'+side]); Wm=sp[max(FMS)][0]; K=Wm.shape[0]
    return Wm.reshape(K,-1).T.contiguous().double(), d['bf_'+side][0].double()      # (N,K),(K,d)
def projector(Wm,tol=1e-6):
    Uw,S,_=torch.linalg.svd(Wm,full_matrices=False); keep=S>S[0]*tol; return Uw[:,keep], int(keep.sum())
def proj(Q,U): U=U.double(); return Q@(Q.T@U)
def e_sub(Q,U):
    U=U.double(); Uc=U-U.mean(0,keepdim=True); off=U-proj(Q,U)
    return dict(e_sub=float(off.norm()/(Uc.norm()+1e-12)),off_rms=float(off.norm()/math.sqrt(U.numel())),cen_rms=float(Uc.norm()/math.sqrt(U.numel())))
def e_aff(X,U,seed=0):
    """affine fit U≈XA+b on half the rows, residual on held-out rows."""
    X=X.double(); U=U.double(); N=len(X); g=torch.Generator(device=X.device).manual_seed(seed); p=torch.randperm(N,generator=g,device=X.device); tr,te=p[:N//2],p[N//2:]
    Xa=torch.cat([X,torch.ones(N,1,device=X.device,dtype=torch.float64)],1)
    G=Xa[tr].T@Xa[tr]; sol=torch.linalg.solve(G+1e-6*torch.trace(G)/G.shape[0]*torch.eye(G.shape[0],device=G.device,dtype=G.dtype),Xa[tr].T@U[tr]); R=U[te]-Xa[te]@sol; Uc=U[te]-U[te].mean(0,keepdim=True)
    return dict(e_aff=float(R.norm()/(Uc.norm()+1e-12)),aff_rms=float(R.norm()/math.sqrt(R.numel())))
def commutation_err(Q,Wm,Fm,Tfun):
    """J = T(W F) - W T(F), for a frozen map T applied to blob vectors (out-of-support warning if far from cloud)."""
    X=(Wm@Fm); TX=Tfun(X.float()).double(); TF=Tfun(Fm.float()).double(); J=TX-Wm@TF
    return dict(comm=float(J.norm()/(TX-TX.mean(0)).norm()),comm_rms=float(J.norm()/math.sqrt(J.numel())))
def neighbor_disc(U,H,W):
    G=U.reshape(H,W,-1); return float(((G[:,1:]-G[:,:-1]).norm(dim=-1).mean()+(G[1:]-G[:-1]).norm(dim=-1).mean())/2)
def var_ret(U,Y): return float(U.double().var(0).sum()/Y.double().var(0).sum())
def interaction_scale(X,Y):
    """RMS of -2 Xc Yc^T (centered interaction) vs raw median cost: the calibration the plan asks for."""
    Xc=X.double()-X.double().mean(0); Yc=Y.double()-Y.double().mean(0); M_=-2*Xc@Yc.T
    return dict(inter_rms=float(M_.pow(2).mean().sqrt()),cost_med=float(torch.cdist(X.double(),Y.double()).pow(2).median()))

# --- observers (canonical) ---
import timm, lpips as lpips_lib
dino=timm.create_model('vit_small_patch14_dinov2.lvd142m',pretrained=True,num_classes=0).to(device).eval()
from torchvision.models import inception_v3, Inception_V3_Weights
_inc=inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1,aux_logits=True).to(device).eval(); _inc.fc=torch.nn.Identity()
for p in list(dino.parameters())+list(_inc.parameters()): p.requires_grad_(False)
_M=torch.tensor([0.485,0.456,0.406],device=device).view(1,3,1,1); _S=torch.tensor([0.229,0.224,0.225],device=device).view(1,3,1,1)
def _stack(pils): return torch.stack([torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/255. for p in pils]).to(device)
@torch.no_grad()
def dino_feats(pils,bs=64):
    out=[]
    for i in range(0,len(pils),bs):
        b=F.interpolate(_stack(pils[i:i+bs]),size=518,mode='bilinear',align_corners=False); out.append(dino((b-_M)/_S).cpu().numpy())
    return np.concatenate(out)
@torch.no_grad()
def inc_feats(pils,bs=64):
    out=[]
    for i in range(0,len(pils),bs):
        b=F.interpolate(_stack(pils[i:i+bs]),size=299,mode='bilinear',align_corners=False); out.append(_inc((b-_M)/_S).cpu().numpy())
    return np.concatenate(out)
lp=lpips_lib.LPIPS(net='alex').to(device).eval()
@torch.no_grad()
def lpips_d(p1,p2):
    t=lambda p:(torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/127.5-1)[None].to(device)
    return float(lp(t(p1),t(p2)))
SCORE_RES=128
def at128(pil): return pil.convert('RGB').resize((SCORE_RES,SCORE_RES))
ref_inc=np.load(f'{OUT}/ref_inc.npy'); ref_dino=np.load(f'{OUT}/ref_dino.npy')
class Manifold:
    def __init__(self,R,k=3,n=1000,seed=0):
        rng=np.random.RandomState(seed); self.R=R[rng.choice(len(R),min(n,len(R)),replace=False)]
        D=np.linalg.norm(self.R[:,None]-self.R[None],axis=-1); D.sort(1); rad=D[:,k]
        keep=rad<=np.median(rad); self.R=self.R[keep]; self.rad=rad[keep]
    def __call__(self,X,bs=256):
        out=[]
        for i in range(0,len(X),bs):
            D=np.linalg.norm(X[i:i+bs,None]-self.R[None],axis=-1); out.append((self.rad[None]/(D+1e-9)).max(1))
        return np.concatenate(out)
MANI_INC=[Manifold(ref_inc,seed=s) for s in range(5)]; MANI_DINO=[Manifold(ref_dino,seed=s) for s in range(5)]
def realism(F_,manis): return np.median(np.stack([m(F_) for m in manis]),0)
def kid(X,Y,n_sub=1000,n_rep=20,seed=0):
    rng=np.random.RandomState(seed); dd=X.shape[1]; m=min(n_sub,len(X),len(Y)); vals=[]
    for _ in range(n_rep):
        x=X[rng.choice(len(X),m,replace=False)]; y=Y[rng.choice(len(Y),m,replace=False)]
        kk=lambda a,b:(a@b.T/dd+1)**3; kxx,kyy,kxy=kk(x,x),kk(y,y),kk(x,y)
        vals.append((kxx.sum()-np.trace(kxx))/(m*(m-1))+(kyy.sum()-np.trace(kyy))/(m*(m-1))-2*kxy.mean())
    return float(np.mean(vals)),float(np.std(vals))
def kid_point(X,Y,seed=0):
    rng=np.random.RandomState(seed); dd=X.shape[1]; m=min(len(X),len(Y)); x=X[:m]; y=Y[rng.choice(len(Y),m,replace=False)]
    kk=lambda a,b:(a@b.T/dd+1)**3; kxx,kyy,kxy=kk(x,x),kk(y,y),kk(x,y)
    return float((kxx.sum()-np.trace(kxx))/(m*(m-1))+(kyy.sum()-np.trace(kyy))/(m*(m-1))-2*kxy.mean())
def boot_ci(x,n=2000,seed=0):
    rng=np.random.RandomState(seed); return np.percentile([np.median(rng.choice(x,len(x))) for _ in range(n)],[2.5,97.5])
MAN_ALL=json.load(open(f'{SCALE}/manifest.json'))['pairs']
KS120=[int(k) for k in pd.read_csv(f'{BASE}/pairs_120/pairs.csv').k]; KSET=set(KS120)
DEV=[int(p_['k']) for p_ in MAN_ALL if int(p_['k']) not in KSET][:24]          # development set: pair-disjoint from the 120 confirmation pairs
print('dev pairs:',DEV); print('confirmation pairs:',len(KS120))

# --- per-pair scoring bundle for a decoded midpoint (human-study proxies) ---
_anc={}
def anchors(k):
    if k in _anc: return _anc[k]
    A,B=pair_imgs(k); rA=to_pil(make_recon(A)); rB=to_pil(make_recon(B)); f=dino_feats([at128(rA),at128(rB)])
    _anc[k]=dict(fA=f[0],fB=f[1],dAB=float(np.linalg.norm(f[0]-f[1])),rA=rA,rB=rB); return _anc[k]
def reencode_field(pil):
    e=cnt_load(pil); ba,bfa,sia=get_gen_inputs(e); sm=build_style_maps(bfa,sia,FMS); L=max(FMS); A=sm[L][0]; Cc=A.shape[0]
    return A.permute(1,2,0).reshape(-1,Cc)
def score_image(k,pil,Z=None,t=None):
    a=anchors(k); im=at128(pil); fi=inc_feats([im])[0]; fd=dino_feats([im])[0]
    r=dict(real_inc=float(realism(fi[None],MANI_INC)[0]),real_dino=float(realism(fd[None],MANI_DINO)[0]),
           dA=float(np.linalg.norm(fd-a['fA'])/a['dAB']),dB=float(np.linalg.norm(fd-a['fB'])/a['dAB']))
    r['place']=r['dA']/(r['dA']+r['dB']+1e-9)
    if Z is not None:
        Xr=reencode_field(pil); r['roundtrip']=float((Xr.double()-Z.double()).norm()/(Z.double().norm()+1e-12))
    return r,fi,fd
_cache_d={}
def get_d(k):
    if k not in _cache_d:
        if len(_cache_d)>64: _cache_d.clear()
        A,B=pair_imgs(k); _cache_d[k]=build_d(cnt_load(A),cnt_load(B))
    return _cache_d[k]
def save_rows(rows,path): pd.DataFrame(rows).to_csv(path,index=False)
def load_rows(path,keys):
    if os.path.exists(path):
        df=pd.read_csv(path); return df.to_dict('records'), set(tuple(r[c] for c in keys) for r in df.to_dict('records'))
    return [],set()
print(f'setup done in {time.time()-t0:.0f}s')
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False; torch.backends.cuda.matmul.allow_tf32=False; torch.backends.cudnn.allow_tf32=False
print('precision: fp32, deterministic cuDNN')


In [ ]:
# ============================================================
# 5. TEXTURE MIXER AT n = 1,000 — canonical scoring, trace-corrected KID, paired leave-one-pair-out jackknife
# ============================================================
SCALE=f'{BASE}/tm_scale1000'; TMC=f'{SCALE}/tmc1000'; JK=f'{BASE}/jackknife'; os.makedirs(JK,exist_ok=True)
REF=ref_inc.astype(np.float64); dd=REF.shape[1]; kk=lambda a,b:(a@b.T/dd+1)**3
K_ref=kk(REF,REF); KYY=float((K_ref.sum()-np.trace(K_ref))/(len(REF)*(len(REF)-1))); del K_ref
def kid_terms(X): kxx=kk(X,X); np.fill_diagonal(kxx,0.0); return kxx,kk(X,REF).mean(1)
def kid_point(X): kxx,r=kid_terms(X); m=len(X); return float(kxx.sum()/(m*(m-1))+KYY-2*r.mean())
def jack(XA,XB):
    """paired leave-one-out via row sums: O(n^2) total"""
    n=len(XA); (kA,rA),(kB,rB)=kid_terms(XA),kid_terms(XB); SA,SB=kA.sum(),kB.sum(); rowA,rowB=kA.sum(1),kB.sum(1); mA,mB=rA.sum(),rB.sum()
    full=(SA/(n*(n-1))-2*mA/n)-(SB/(n*(n-1))-2*mB/n); m=n-1
    loo=((SA-2*rowA)/(m*(m-1))-2*(mA-rA)/m)-((SB-2*rowB)/(m*(m-1))-2*(mB-rB)/m)
    se=float(np.sqrt((n-1)/n*((loo-loo.mean())**2).sum())); return full,full-1.96*se,full+1.96*se
RULES=['lerp','mean','ot_r0.03']; KS1000=sorted(int(os.path.basename(f)[:4]) for f in glob.glob(f'{TMC}/lerp/*.png')); print(len(KS1000),'pairs')
def score_dir(name,d,ks):
    fp=f'{JK}/scores_{name}.csv'; fq=f'{JK}/feats_{name}.npy'
    if os.path.exists(fp) and os.path.exists(fq): return pd.read_csv(fp),np.load(fq)
    rows,F=[],[]; t0=time.time()
    for i,k in enumerate(ks):
        s,fi,fd=score_image(k,Image.open(f'{d}/{k:04d}.png').convert('RGB')); rows.append(dict(k=k,**s)); F.append(fi)
        if i%200==0: print(name,i,f'{time.time()-t0:.0f}s')
    S=pd.DataFrame(rows); F=np.stack(F).astype(np.float64); S.to_csv(fp,index=False); np.save(fq,F); return S,F
R={r:score_dir(f'tm1000_{r}',f'{TMC}/{r}',KS1000) for r in RULES}
T=pd.DataFrame({r:dict(KID=kid_point(F),RInc=float(S.real_inc.median()),RDin=float(S.real_dino.median()),Bal=float((S.place-0.5).abs().median()),Place=float(S.place.median())) for r,(S,F) in R.items()}).T
print('=== Texture Mixer coupling, n=%d, canonical scoring ==='%len(KS1000)); print(T.round(4).to_string())
J=[]
for r,name in [('mean','TMkdKIDmean'),('ot_r0.03','TMkdKIDot')]:
    d,lo,hi=jack(R[r][1],R['lerp'][1]); J.append(dict(macro=name,comparison=f'TM {r} − lerp (n=1000)',dKID=d,lo=lo,hi=hi,excl0=lo*hi>0))
d,lo,hi=jack(R['ot_r0.03'][1],R['mean'][1]); J.append(dict(macro='TMkdKIDotVsMean',comparison='TM entropic − mean (n=1000)',dKID=d,lo=lo,hi=hi,excl0=lo*hi>0))
J=pd.DataFrame(J); J.to_csv(f'{JK}/tm1000_jackknife.csv',index=False); print(J.round(4).to_string(index=False))
pv=R['mean'][0].merge(R['lerp'][0],on='k',suffixes=('_m','_l')); print('paired realism (DINO) mean − lerp: median %+.4f p=%.2g'%((pv.real_dino_m-pv.real_dino_l).median(),wilcoxon(pv.real_dino_m-pv.real_dino_l).pvalue))
print('\n% macros, Texture Mixer at n=1000')
for r,s in [('lerp','lerp'),('mean','mean'),('ot_r0.03','ot')]: print(f'\\newcommand{{\\TMkKID{s}}}{{{T.loc[r,"KID"]:.4f}}}\\newcommand{{\\TMkRInc{s}}}{{{T.loc[r,"RInc"]:.3f}}}\\newcommand{{\\TMkBal{s}}}{{{T.loc[r,"Bal"]:.3f}}}')
for _,r in J.iterrows(): print(f'\\newcommand{{\\{r.macro}}}{{{r.dKID:+.4f}}}\\newcommand{{\\{r.macro}CI}}{{[{r.lo:+.4f},{r.hi:+.4f}]}}   % {r.comparison}: {"excludes 0" if r.excl0 else "INCLUDES 0"}')
print(f'\\newcommand{{\\TMkN}}{{{len(KS1000)}}}')


In [ ]:
# ============================================================
# 6. PAIR-CONDITIONED DIAGNOSTIC — pair retrieval: for each midpoint, rank its own (A,B) pair among all pairs by
#    d(mid, recon A_j) + d(mid, recon B_j) in DINOv2 space. A generic plausible texture retrieves nobody; a real intermediate retrieves its pair.
#    Controls: source copy (retrieves A but the pair only through A), target copy, target mean.
# ============================================================
def retrieval_table(ks,dirs,tag):
    fA=np.stack([anchors(k)['fA'] for k in ks]); fB=np.stack([anchors(k)['fB'] for k in ks]); out={}
    for name,d in dirs.items():
        if not os.path.isdir(d): continue
        fp=f'{JK}/dino_{tag}_{name}.npy'
        if os.path.exists(fp): F=np.load(fp)
        else: F=dino_feats([at128(Image.open(f'{d}/{k:04d}.png').convert('RGB')) for k in ks]); np.save(fp,F)
        D=np.linalg.norm(F[:,None]-fA[None],axis=-1)+np.linalg.norm(F[:,None]-fB[None],axis=-1)      # (n mids, n pairs) joint distance
        own=D[np.arange(len(ks)),np.arange(len(ks))]; rank=(D<own[:,None]).sum(1)+1
        DA=np.linalg.norm(F[:,None]-fA[None],axis=-1); DB=np.linalg.norm(F[:,None]-fB[None],axis=-1)
        rA=(DA<DA[np.arange(len(ks)),np.arange(len(ks))][:,None]).sum(1)+1; rB=(DB<DB[np.arange(len(ks)),np.arange(len(ks))][:,None]).sum(1)+1
        out[name]=dict(pair_MRR=float((1/rank).mean()),pair_top1=float((rank==1).mean()),A_MRR=float((1/rA).mean()),B_MRR=float((1/rB).mean()),n=len(ks)); out[name]['_rank']=rank
    return out
KS120=sorted(int(k) for k in pd.read_csv(f'{BASE}/pairs_120/pairs.csv').k); KS200=[int(p_['k']) for p_ in MAN_ALL[:200]]
gsym=next((p for p in [f'{SCALE}/gpt_sym',f'{SCALE}/gpt_symmetric',f'{SCALE}/gpt_p1'] if os.path.isdir(p)),f'{SCALE}/gpt_sym')
D120={'field_linear':f'{SCALE}/pixel_linear','primitive_linear':f'{SCALE}/texton_linear','gaussian':f'{SCALE}/gaussian_w2','AAT':f'{SCALE}/ot','hard':f'{SCALE}/exact','texture_mixer':f'{SCALE}/mid','gpt_directed':f'{SCALE}/gpt','gpt_symmetric':gsym,
      'tm_mean_coupling':f'{SCALE}/tmc/mean','tm_entropic_coupling':f'{SCALE}/tmc/ot_r0.03','ctrl_source_copy':f'{SCALE}/source_copy','ctrl_target_copy':f'{SCALE}/target_copy','ctrl_target_mean':f'{SCALE}/target_mean'}
R120=retrieval_table(KS120,D120,'g'); T120=pd.DataFrame({n:{k:v for k,v in r.items() if not k.startswith('_')} for n,r in R120.items()}).T
print('=== pair retrieval, 120 pairs (chance MRR ≈ %.3f) ==='%float(np.mean(1/np.arange(1,121)))); print(T120.round(3).to_string())
D200={'field_linear_cp':f'{OUT}/eqsample/linear_cp_t0.5','hard':f'{OUT}/eqsample/hard_t0.5','entropic_0.005':f'{OUT}/eqsample/ot_e0.005_t0.5','entropic_0.05':f'{OUT}/eqsample/ot_e0.05_t0.5','AAT_4096':f'{SCALE}/ot','ctrl_target_mean':f'{SCALE}/target_mean','ctrl_source_copy':f'{SCALE}/source_copy'}
R200=retrieval_table(KS200,D200,'eq'); T200=pd.DataFrame({n:{k:v for k,v in r.items() if not k.startswith('_')} for n,r in R200.items()}).T
print('\n=== pair retrieval, matched-rendering 200 pairs (chance MRR ≈ %.3f) ==='%float(np.mean(1/np.arange(1,201)))); print(T200.round(3).to_string())
T120.to_csv(f'{JK}/pair_retrieval_120.csv'); T200.to_csv(f'{JK}/pair_retrieval_200.csv')
def paired_p(a,b,R): x=1/R[a]['_rank']-1/R[b]['_rank']; return float(np.median(x)),float(wilcoxon(x).pvalue) if np.abs(x).sum()>0 else 1.0
for a,b,R,lab in [('AAT','field_linear',R120,'120'),('AAT','hard',R120,'120'),('AAT','texture_mixer',R120,'120'),('AAT','gpt_directed',R120,'120'),('entropic_0.05','field_linear_cp',R200,'200'),('entropic_0.05','hard',R200,'200')]:
    if a in R and b in R: m,p=paired_p(a,b,R); print(f'paired reciprocal-rank {a} − {b} ({lab}): median {m:+.3f}, Wilcoxon p={p:.2g}')
print('\n% macros, pair retrieval (MRR of own pair)')
for n,s in [('field_linear','pix'),('gaussian','gauss'),('AAT','ot'),('hard','hard'),('texture_mixer','tm'),('gpt_directed','gpt'),('ctrl_target_mean','mean'),('ctrl_source_copy','srccopy')]:
    if n in T120.index: print(f'\\newcommand{{\\PRmrr{s}}}{{{T120.loc[n,"pair_MRR"]:.3f}}}\\newcommand{{\\PRtop{s}}}{{{100*T120.loc[n,"pair_top1"]:.0f}\\%}}')
for n,s in [('field_linear_cp','linear'),('hard','hard'),('entropic_0.05','ot'),('AAT_4096','paper')]:
    if n in T200.index: print(f'\\newcommand{{\\PReqmrr{s}}}{{{T200.loc[n,"pair_MRR"]:.3f}}}')
